## How to use: Checklist

In this notebook, we demonstrate the use step by step of different modules of Checklist.

By: William Eduardo Peñaranda-Arévalo --> Analyst of SGC.

### 1. Module imports

In this section we load checklist modules to be able to use in main script file. It's required to run only one time bellow cell due to every new execution change current execution path. In other case it's required to restart jupyter variables.

In [1]:
## Trick to change current location
%cd ..
import numpy as np
import pandas as pd
import time
from datetime import datetime, UTC
from tqdm import tqdm
from src.config import settings
from src.data_access.slinktool_executor import SlinktoolExecutor
from src.data_access.load_data import StationRepository
from src.services.delay_calculator import DelayCalculator
from src.services.station_analysis import StationAnalysis
from src.utilities.data_formatter import SiigeoFormatter, OtherStationsFormatter
from src.utilities.slinktool_formatter import SlinktoolFormatter
from src.utilities.table_formatter import TableFormatter
%pwd
logger = settings.get_logger(__name__)

/home/wpenaranda/error_404/check_list


### 2. Load Station data from SGC database

Here, we load the information of stations provided by SGC storaged in data folder.  

In [2]:
siigeo_data = StationRepository.get_siigeo_stations()
shared_data = StationRepository.get_shared_stations()
inter_data = StationRepository.get_inter_stations()

12:08:16 [INFO] Stations from Siigeo file succesfully load.
12:08:16 [INFO] Shared stations succesfully load.
12:08:16 [INFO] International stations succesfully load.


Next, let's take a look of the station information. 

First we load Siigeo information: 

In [3]:
siigeo_data.head()

,IDENTIFICADOR,ESTADO ESTACIÓN,CÓDIGO LOCALIZACION,ESTADO COD LOC.,TIPO DE TRANSMISIÓN,TIPO DE ESTACIÓN,TIPO DE ADQUISICIÓN,network
0,ACH1,ACTIVA,00,Activa,Radio,Permanente,Tiempo Real,CM
1,ACH1,ACTIVA,10,Activa,Radio,Permanente,Tiempo Real,CM
2,ACH6,ACTIVA,30,Activa,Radio,Permanente,Tiempo Real,CM
3,AGCC,ACTIVA,00,Activa,Celular,Permanente,Tiempo Real,CM
4,APAC,ACTIVA,00,Activa,Satelital LEO,Permanente,Tiempo Real,CM


Second we load information compiled for shared national stations: 

In [4]:
shared_data.head()

,IDENTIFICADOR,nombre,CODIGO RED,CODIGO SENSOR,CODIGO CANAL,ESTADO ESTACION,TIPO DE TRANSMISION,TIPO DE ADQUISICION
0,GCUF,Cufiño(Galeras),OP,HHZ,00,activa,internet,tiempo real
1,LMES,Mesa de Los Santos,OS,HNZ,10,activa,internet,tiempo real
2,VNAZ,Vivero Nazareth,OS,HNZ,10,activa,internet,tiempo real
3,PBLA,Peñas Blancas,UQ,HHZ,00,activa,internet,tiempo real
4,RIBER,Ribera,UQ,EHZ,20,activa,internet,tiempo real


and finally, we load international stations:

In [5]:
inter_data.head()

,IDENTIFICADOR,nombre,pais,CODIGO RED,CODIGO SENSOR,CODIGO CANAL,ESTADO ESTACION,TIPO DE TRANSMISION,TIPO DE ADQUISICION
0,ACPR,Escuela Internacional de Aruba,Paises Bajos,PR,HHZ,00,activa,internet,tiempo real
1,ACON,Acoyapa,Nicaragua,NU,HHZ,,activa,internet,tiempo real
2,ACOT,,Ecuador,EC,HNZ,,inactiva,internet,tiempo real
3,ALCE,"Alcedo, Galapagos",Ecuador,EC,HHZ,,activa,internet,tiempo real
4,ANWB,North Barbuda Island,Antigua y Barbuda,CU,BHZ,00,activa,internet,tiempo real


### 3. Adjust of SGC station information

In this section, we need to modify format of the input information to unify column names and format used in every value of the tables in a only dataframe.

In [6]:
data = {'siigeo': siigeo_data, 'shared': shared_data, 'inter': inter_data}
format_data = {}
for key, df in data.items():
    strategy = SiigeoFormatter() if key == 'siigeo' else OtherStationsFormatter()
    df = strategy.rename_cols(df)
    df = strategy.assign_data_types(df)
    df = strategy.infer_channel(df)
    df = strategy.filter_data(df)
    df = strategy.default_connection(df)
    format_data[key] = df     

siigeo_data = format_data['siigeo']
shared_data = format_data['shared']
inter_data = format_data['inter']
all_data = pd.concat([siigeo_data, shared_data, inter_data], ignore_index=True)
all_data.head()

12:08:26 [INFO] Dataframe formatted succesfully. n=208
12:08:26 [INFO] Dataframe formatted succesfully. n=20
12:08:26 [INFO] Dataframe formatted succesfully. n=65


,network,stid,loc,channel,type_transmission,last_connection
0,CM,ACH1,00,HHZ,radio,None
1,CM,ACH1,10,HNZ,radio,None
2,CM,ACH6,30,HHZ,radio,None
3,CM,AGCC,00,HHZ,celular,None
4,CM,APAC,00,HHZ,satelital leo,None


### 4. Request connectivity information of SGC stations

Next, we use a Seiscomp client to request the last connection datetime for SGC stations.

In [7]:
port = SlinktoolExecutor.get_port('91')
comm = SlinktoolExecutor.build_command('91', port)
output = SlinktoolExecutor.request_info(comm)
solution = SlinktoolFormatter.extract_info(output)
slink_data = SlinktoolFormatter.build_dataframe(solution)
slink_data = SlinktoolFormatter.select_interest_channels(slink_data)
slink_data = SlinktoolFormatter.assign_transmission(slink_data, all_data)
slink_data = SlinktoolFormatter.order_cols(slink_data)
slink_data.head()

12:08:29 [INFO] Port stablished succesfully.
12:08:29 [INFO] Sucessful information request from slinktool.
12:08:29 [INFO] Number of sensors to be processed: 1500
12:08:29 [INFO] Success!!
12:08:29 [INFO] Dataframe created succesfully!!
12:08:29 [INFO] Selection of the 795 field sensors completed succesfully!!
12:08:29 [INFO] Transmission type assigned succesfully!!
12:08:29 [INFO] Dataframe formatted succesfully according to SGC data!!


,network,stid,loc,channel,type_transmission,last_connection
0,BR,CZSB,<NA>,HHE,internet,2026-08-08 17:08:18.250
1,BR,CZSB,<NA>,HHN,internet,2026-08-08 17:08:16.910
2,BR,CZSB,<NA>,HHZ,internet,2026-08-08 17:08:19.470
3,CM,ACH1,00,HHE,radio,2026-06-12 04:48:04.710
4,CM,ACH1,00,HHN,radio,2026-06-12 04:48:04.890


### 5. Compute delays of SGC stations

Later, we compute last datetime connection and delays for all SGC stations.  

In [8]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    final_data = DelayCalculator.add_stations_untracked(all_data, slink_data)

actual_time = datetime.now(UTC).replace(tzinfo=None)
final_data = DelayCalculator.compute_delays(final_data, actual_time, 'last_connection')
final_data = DelayCalculator.format_delays(final_data, delay_column='delay')
final_data.head()

12:08:33 [INFO] Adding stations not tracked by slinktool succesfully!!
12:08:33 [INFO] Delays succesfully computed!!
12:08:33 [INFO] Delays succesfully formatted!!


,network,stid,loc,channel,type_transmission,last_connection,delay,delay_
0,BR,CZSB,<NA>,HHE,internet,2026-08-08 17:08:18.250000,0 days 00:00:15.262464,0.0d 0.0h 0.0m 15.0s
1,BR,CZSB,<NA>,HHN,internet,2026-08-08 17:08:16.910000,0 days 00:00:16.602464,0.0d 0.0h 0.0m 16.0s
2,BR,CZSB,<NA>,HHZ,internet,2026-08-08 17:08:19.470000,0 days 00:00:14.042464,0.0d 0.0h 0.0m 14.0s
3,CM,ACH1,00,HHE,radio,2026-06-12 04:48:04.710000,57 days 12:20:28.802464,57.0d 12.0h 20.0m 28.0s
4,CM,ACH1,00,HHN,radio,2026-06-12 04:48:04.890000,57 days 12:20:28.622464,57.0d 12.0h 20.0m 28.0s


### 6. Re validate station status

In this section, we can perform a second validation of station connectivity status after a certain period of time. By default, time is set as 30 seconds.

In [9]:
dt = 30
with tqdm(total=dt, desc=f"Waiting {dt} seconds to execute a new validation") as pbar:
    for _ in range(dt):
        time.sleep(1)
        pbar.update(1)

output = SlinktoolExecutor.request_info(comm)
solution = SlinktoolFormatter.extract_info(output)
slink_data = SlinktoolFormatter.build_dataframe(solution)
slink_data = SlinktoolFormatter.select_interest_channels(slink_data)
slink_data = SlinktoolFormatter.assign_transmission(slink_data, all_data)
slink_data = SlinktoolFormatter.order_cols(slink_data)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    final_data2 = DelayCalculator.add_stations_untracked(all_data, slink_data)

actual_time = datetime.now(UTC).replace(tzinfo=None)
final_data2 = DelayCalculator.compute_delays(final_data2, actual_time, 'last_connection')
final_data2 = DelayCalculator.format_delays(final_data2, delay_column='delay')

Waiting 30 seconds to execute a new validation: 100%|██████████| 30/30 [00:30<00:00,  1.00s/it]
12:09:07 [INFO] Sucessful information request from slinktool.
12:09:07 [INFO] Number of sensors to be processed: 1500
12:09:07 [INFO] Success!!
12:09:07 [INFO] Dataframe created succesfully!!
12:09:07 [INFO] Selection of the 795 field sensors completed succesfully!!
12:09:07 [INFO] Transmission type assigned succesfully!!
12:09:07 [INFO] Dataframe formatted succesfully according to SGC data!!
12:09:07 [INFO] Adding stations not tracked by slinktool succesfully!!
12:09:07 [INFO] Delays succesfully computed!!
12:09:07 [INFO] Delays succesfully formatted!!


### 7. Determine station status for SGC stations

Later we determine status for each station.

In [10]:
# Extract delay columns
delay1 = final_data['delay'].dt.total_seconds().fillna(np.nan).to_numpy()
delay2 = final_data2['delay'].dt.total_seconds().fillna(np.nan).to_numpy()
status = StationAnalysis.get_station_status(delay1, delay2, dt)
final_data2['status'] = status  
final_data2.head()

12:09:15 [INFO] Station status determined succesfully!!


,network,stid,loc,channel,type_transmission,last_connection,delay,delay_,status
0,BR,CZSB,<NA>,HHE,internet,2026-08-08 17:08:59.450000,0 days 00:00:08.267218,0.0d 0.0h 0.0m 8.0s,Online
1,BR,CZSB,<NA>,HHN,internet,2026-08-08 17:08:58.110000,0 days 00:00:09.607218,0.0d 0.0h 0.0m 9.0s,Online
2,BR,CZSB,<NA>,HHZ,internet,2026-08-08 17:08:56.550000,0 days 00:00:11.167218,0.0d 0.0h 0.0m 11.0s,Online
3,CM,ACH1,00,HHE,radio,2026-06-12 04:48:04.710000,57 days 12:21:03.007218,57.0d 12.0h 21.0m 3.0s,Offline
4,CM,ACH1,00,HHN,radio,2026-06-12 04:48:04.890000,57 days 12:21:02.827218,57.0d 12.0h 21.0m 2.0s,Offline


### 8. Calculate performance metrics for SGC stations

If it's required, we can compute some metrics about station connectivity as presented below: 

In [11]:
stats = StationAnalysis.sensor_metrics(final_data2) 
print(stats)

12:09:18 [INFO] Sensor metrics computed succesfully!!


{'online_stations': 157, 'total_stations': 227, 'online_seismo': 97, 'total_seismo': 145, 'online_accel': 90, 'total_accel': 121, 'online_strong': 6, 'total_strong': 6, 'online_satellital': 45, 'total_satellital': 50, 'online_cellphone': 61, 'total_cellphone': 74, 'online_internet': 55, 'total_internet': 100, 'online_radio': 1, 'total_radio': 3}


### 9. Format results

Finally, we adjust results' structure according to Checklist document stablish in SGC monitoring room.

In [12]:
from colorama import Fore
seismo_stations, accel_stations, inter_stations = TableFormatter.order_stations_checklist(final_data2, list(inter_data['stid']))
inter_stations = TableFormatter.extract_inter_stations(inter_data, inter_stations, settings.stations.INTER_PATTERN_CHANNELS)

# Format dataframe results to be displayed in terminal
_seismo_stations = TableFormatter.get_display_table(seismo_stations, 'Seismological', mode=False)
_seismo_stations['status'] = _seismo_stations['status'].apply(TableFormatter.format_status)
_accel_stations = TableFormatter.get_display_table(accel_stations, 'Acelerographic', mode=False)
_accel_stations['status'] = _accel_stations['status'].apply(TableFormatter.format_status)
_inter_stations = TableFormatter.get_display_table(inter_stations, 'International', mode=False)
_inter_stations['status'] = _inter_stations['status'].apply(TableFormatter.format_status)
logger.info(Fore.GREEN + "Station status formatted succesfully!!" + Fore.RESET)

12:10:19 [INFO] Station data ordered according to checklist!!


12:10:19 [INFO] succesfully extracted international stations!!
12:10:19 [INFO] Seismological data succesfully formatted to display!!
12:10:19 [INFO] Acelerographic data succesfully formatted to display!!
12:10:19 [INFO] International data succesfully formatted to display!!
12:10:19 [INFO] Station status formatted succesfully!!


In [13]:
print(_seismo_stations)

    network   stid loc channel             last_connection             status
0        CM   ACH1  00     HHZ  2026-06-12 04:48:05.130000  Offline
1        CM   ACH6  30     HHZ  2026-08-08 17:09:00.540000   Online
2        CM   AGCC  00     HHZ  2026-08-08 15:05:46.930000  Offline
3        CM   APAC  00     HHZ  2026-08-08 17:09:04.358300   Online
4        CM  ARAMC  00     HHZ  2026-08-08 17:09:03.180000   Online
..      ...    ...  ..     ...                         ...                ...
104      CM   SOTO  00     HHZ  2026-08-08 17:09:01.030000   Online
105      CM   PALO  00     HHZ                        None      N/A
106      CM   BODP  00     HHZ  2026-08-08 17:08:57.205000   Online
107      OP   GCUF  00     HHZ  2026-08-08 17:08:51.810000   Online
108      CM   GUAP  00     HHZ  2026-08-08 17:08:57.030000   Online

[109 rows x 6 columns]


In [14]:
print(_accel_stations)

    network   stid loc channel             last_connection             status
0        CM   ACH1  10     HNZ  2026-06-12 04:48:07.085000  Offline
1        CM   APAC  10     HNZ  2026-08-08 17:09:05.984500   Online
2        CM  ARAUC  10     HNZ  2026-08-08 17:09:04.850000   Online
3        CM   ARGC  10     HNZ  2026-08-08 17:08:54.689500   Online
4        CM   ARGC  40     HLZ  2026-08-08 17:09:06.139500   Online
..      ...    ...  ..     ...                         ...                ...
118      CM  VALLC  10     HNZ  2026-08-08 17:09:03.485000   Online
119      CM    VIL  10     HNZ  2026-08-08 17:09:03.284500   Online
120      CM    VIL  40     HLZ  2026-08-08 17:09:03.144500   Online
121      OS   VNAZ  10     HNZ                        None      N/A
122      CM   YPLC  10     HNZ  2026-08-08 17:09:02.210000   Online

[123 rows x 6 columns]


In [15]:
print(_inter_stations)

   network   stid   loc channel             last_connection             status
0       NU   ACON  <NA>     HHZ  2026-08-08 17:09:03.840000   Online
1       EC   ACOT           HNZ                        None      N/A
2       PR   ACPR    00     HHZ  2026-08-08 17:08:50.695000   Online
3       EC   ALCE           HHZ                        None      N/A
4       CU   ANWB    00     BHZ  2026-08-08 17:08:59.819500   Online
..     ...    ...   ...     ...                         ...                ...
60      RP   TOTI  <NA>     HHZ  2026-08-08 17:09:06.050000   Online
61      EC   TULM  <NA>     HHZ  2026-08-08 17:09:06.808300   Online
62      RP    UPA  <NA>     HHZ  2026-07-28 14:56:04.208300  Offline
63      RP   UPD2           HHZ                        None      N/A
64      RP  YAVI3  <NA>     SHZ  2026-08-08 17:09:04.580000   Online

[65 rows x 6 columns]
